# DS3/GSE182109 — dataset-level quality control

This notebook reproduces the DS3 filtering rules used in the study.

## Cell filters

Primary cell retention requires:

- detected genes > 500;
- detected genes < 8,000;
- total counts > 1,000;
- total counts < 60,000;
- mitochondrial fraction < 12%.

A final total-count ceiling of 45,000 is then applied, followed by
retention of genes detected in at least 10 final cells.

## Expected dimensions

- input: 264,951 cells × 38,224 genes
- after primary cell filtering: 219,418 cells × 38,224 genes
- final post-QC: 218,735 cells × 30,284 genes

Raw counts remain in `adata.X` and are copied to
`adata.layers["counts"]` only after filtering to reduce peak memory use.
No normalization or log transformation is performed.

Input:

`data/interim/DS3_GSE182109/GSE182109_merged_rawcounts.h5ad`

Output:

`data/processed/DS3_GSE182109/GSE182109_postQC.h5ad`

In [ ]:
import json
import os
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

print("Python:", sys.version)
print("scanpy:", sc.__version__)
print("pandas:", pd.__version__)

In [ ]:
def find_repository_root(start: Path) -> Path:
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start


PROJECT_ROOT = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", find_repository_root(Path.cwd()))
).expanduser().resolve()

INPUT_H5AD = Path(
    os.environ.get(
        "GSE182109_MERGED_H5AD",
        PROJECT_ROOT
        / "data"
        / "interim"
        / "DS3_GSE182109"
        / "GSE182109_merged_rawcounts.h5ad",
    )
).expanduser().resolve()

OUTPUT_H5AD = Path(
    os.environ.get(
        "GSE182109_POSTQC_H5AD",
        PROJECT_ROOT
        / "data"
        / "processed"
        / "DS3_GSE182109"
        / "GSE182109_postQC.h5ad",
    )
).expanduser().resolve()

AUDIT_DIR = (
    PROJECT_ROOT / "results" / "qc" / "DS3_GSE182109" / "00b_qc"
)
FIGURE_DIR = AUDIT_DIR / "figures"

EXPECTED_INPUT_SHAPE = (264_951, 38_224)
EXPECTED_AFTER_PRIMARY_FILTER = 219_418
EXPECTED_OUTPUT_SHAPE = (218_735, 30_284)

MIN_GENES_EXCLUSIVE = 500
MAX_GENES_EXCLUSIVE = 8_000
MIN_COUNTS_EXCLUSIVE = 1_000
MAX_COUNTS_PRIMARY_EXCLUSIVE = 60_000
MAX_MT_PERCENT_EXCLUSIVE = 12.0
FINAL_MAX_COUNTS_INCLUSIVE = 45_000
FINAL_MIN_CELLS_PER_GENE = 10

OUTPUT_H5AD.parent.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Input:", INPUT_H5AD)
print("Output:", OUTPUT_H5AD)

if not INPUT_H5AD.exists():
    raise FileNotFoundError(
        f"Merged input not found: {INPUT_H5AD}\n"
        "Run 00a_ds3_build_raw_anndata.ipynb first or set "
        "GSE182109_MERGED_H5AD."
    )

In [ ]:
def matrix_values(matrix):
    return matrix.data if sp.issparse(matrix) else np.asarray(matrix).ravel()


def assert_raw_counts(matrix, label: str):
    values = matrix_values(matrix)
    if values.size == 0:
        raise ValueError(f"{label} contains no nonzero values.")
    if not np.isfinite(values).all():
        raise ValueError(f"{label} contains non-finite values.")
    if np.any(values < 0):
        raise ValueError(f"{label} contains negative values.")
    if not np.equal(values, np.floor(values)).all():
        raise ValueError(f"{label} is not an integer raw-count matrix.")


def save_current_figure(path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close("all")

In [ ]:
adata = sc.read_h5ad(INPUT_H5AD)

if adata.shape != EXPECTED_INPUT_SHAPE:
    raise AssertionError(
        f"Expected input shape {EXPECTED_INPUT_SHAPE}, observed {adata.shape}"
    )

required_obs = {"gsm", "sample", "group", "patient", "region", "sample_key"}
missing_obs = sorted(required_obs - set(adata.obs.columns))
if missing_obs:
    raise KeyError(f"Required metadata columns are missing: {missing_obs}")

if not adata.obs_names.is_unique:
    raise AssertionError("Cell identifiers are not unique.")
if not adata.var_names.is_unique:
    raise AssertionError("Gene identifiers are not unique.")

if sp.issparse(adata.X):
    adata.X = adata.X.tocsr()
else:
    adata.X = sp.csr_matrix(adata.X)

assert_raw_counts(adata.X, "adata.X")
print(adata)

In [ ]:
adata.var["feature_name"] = adata.var_names.astype(str)
feature_upper = adata.var["feature_name"].str.upper()

adata.var["mt"] = feature_upper.str.startswith("MT-")
adata.var["ribo"] = feature_upper.str.startswith(("RPS", "RPL"))
adata.var["hb"] = feature_upper.str.match(r"^HB[ABDEGQMZ]")

sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt", "ribo", "hb"],
    percent_top=[50, 100, 200, 500],
    log1p=True,
    inplace=True,
)

required_metrics = [
    "n_genes_by_counts",
    "total_counts",
    "pct_counts_mt",
    "pct_counts_ribo",
    "pct_counts_hb",
]
missing_metrics = [
    column for column in required_metrics if column not in adata.obs
]
if missing_metrics:
    raise RuntimeError(
        f"QC metric calculation failed: missing {missing_metrics}"
    )

print("Mitochondrial genes:", int(adata.var["mt"].sum()))
print(adata.obs[required_metrics].describe())

In [ ]:
primary_masks = pd.DataFrame(index=adata.obs_names)
primary_masks["pass_min_genes"] = (
    adata.obs["n_genes_by_counts"] > MIN_GENES_EXCLUSIVE
)
primary_masks["pass_max_genes"] = (
    adata.obs["n_genes_by_counts"] < MAX_GENES_EXCLUSIVE
)
primary_masks["pass_min_counts"] = (
    adata.obs["total_counts"] > MIN_COUNTS_EXCLUSIVE
)
primary_masks["pass_primary_max_counts"] = (
    adata.obs["total_counts"] < MAX_COUNTS_PRIMARY_EXCLUSIVE
)
primary_masks["pass_max_mt"] = (
    adata.obs["pct_counts_mt"] < MAX_MT_PERCENT_EXCLUSIVE
)
primary_masks["pass_primary_QC"] = primary_masks.all(axis=1)

primary_summary = pd.DataFrame(
    {
        "rule": primary_masks.columns,
        "cells_passing": [
            int(primary_masks[column].sum())
            for column in primary_masks
        ],
        "cells_failing": [
            int((~primary_masks[column]).sum())
            for column in primary_masks
        ],
    }
)
primary_summary.to_csv(
    AUDIT_DIR / "GSE182109_primary_QC_rule_summary.tsv",
    sep="\t",
    index=False,
)

adata_primary = adata[
    primary_masks["pass_primary_QC"].to_numpy()
].copy()

if adata_primary.n_obs != EXPECTED_AFTER_PRIMARY_FILTER:
    raise AssertionError(
        "Primary DS3 filtering did not reproduce the original count. "
        f"Expected {EXPECTED_AFTER_PRIMARY_FILTER}, observed "
        f"{adata_primary.n_obs}."
    )

print("After primary cell filtering:", adata_primary.shape)
primary_summary

In [ ]:
before_sample_counts = (
    adata.obs.groupby("sample", observed=True)
    .size()
    .rename("cells_before_QC")
)

final_cell_mask = (
    adata_primary.obs["total_counts"]
    <= FINAL_MAX_COUNTS_INCLUSIVE
)
adata_qc = adata_primary[final_cell_mask.to_numpy()].copy()

# The historical Scanpy filter_cells call added an n_counts column.
adata_qc.obs["n_counts"] = adata_qc.obs["total_counts"].astype(
    np.float32
)

sc.pp.filter_genes(
    adata_qc,
    min_cells=FINAL_MIN_CELLS_PER_GENE,
)

after_sample_counts = (
    adata_qc.obs.groupby("sample", observed=True)
    .size()
    .rename("cells_after_QC")
)

sample_retention = pd.concat(
    [before_sample_counts, after_sample_counts],
    axis=1,
).fillna(0)
sample_retention["cells_removed"] = (
    sample_retention["cells_before_QC"]
    - sample_retention["cells_after_QC"]
)
sample_retention["retention_fraction"] = (
    sample_retention["cells_after_QC"]
    / sample_retention["cells_before_QC"]
)
sample_retention.to_csv(
    AUDIT_DIR / "GSE182109_QC_retention_by_sample.tsv",
    sep="\t",
)

if adata_qc.shape != EXPECTED_OUTPUT_SHAPE:
    raise AssertionError(
        "DS3 QC did not reproduce the reported output dimensions. "
        f"Expected {EXPECTED_OUTPUT_SHAPE}, observed {adata_qc.shape}."
    )

assert_raw_counts(adata_qc.X, "post-QC adata.X")

# Copy counts only after filtering to avoid doubling the full raw object.
adata_qc.layers["counts"] = adata_qc.X.copy()
assert_raw_counts(
    adata_qc.layers["counts"],
    'post-QC adata.layers["counts"]',
)

print("Final post-QC object:", adata_qc)
sample_retention

In [ ]:
condition_composition = (
    adata_qc.obs.groupby("group", observed=True)
    .size()
    .rename("cells")
    .reset_index()
    .sort_values("group")
)
condition_composition.to_csv(
    AUDIT_DIR / "GSE182109_postQC_condition_composition.tsv",
    sep="\t",
    index=False,
)
condition_composition

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
    show=False,
)
save_current_figure(FIGURE_DIR / "GSE182109_QC_metrics_before_filtering.png")

sc.pl.violin(
    adata_qc,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
    show=False,
)
save_current_figure(FIGURE_DIR / "GSE182109_QC_metrics_after_filtering.png")

sc.pl.scatter(
    adata,
    x="total_counts",
    y="n_genes_by_counts",
    color="pct_counts_mt",
    show=False,
)
save_current_figure(FIGURE_DIR / "GSE182109_scatter_before_filtering.png")

sc.pl.scatter(
    adata_qc,
    x="total_counts",
    y="n_genes_by_counts",
    color="pct_counts_mt",
    show=False,
)
save_current_figure(FIGURE_DIR / "GSE182109_scatter_after_filtering.png")

In [ ]:
qc_summary = {
    "dataset_id": "DS3_GSE182109",
    "geo_accession": "GSE182109",
    "input_cells": int(adata.n_obs),
    "input_genes": int(adata.n_vars),
    "post_primary_filter_cells": int(adata_primary.n_obs),
    "post_QC_cells": int(adata_qc.n_obs),
    "post_QC_genes": int(adata_qc.n_vars),
    "retention_fraction": float(adata_qc.n_obs / adata.n_obs),
    "min_genes_exclusive": MIN_GENES_EXCLUSIVE,
    "max_genes_exclusive": MAX_GENES_EXCLUSIVE,
    "min_counts_exclusive": MIN_COUNTS_EXCLUSIVE,
    "primary_max_counts_exclusive": MAX_COUNTS_PRIMARY_EXCLUSIVE,
    "max_mt_percent_exclusive": MAX_MT_PERCENT_EXCLUSIVE,
    "final_max_counts_inclusive": FINAL_MAX_COUNTS_INCLUSIVE,
    "final_min_cells_per_gene": FINAL_MIN_CELLS_PER_GENE,
    "raw_counts_in_X": True,
    "raw_counts_layer": "counts",
    "normalization_applied": False,
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "output_h5ad": str(OUTPUT_H5AD),
}

adata_qc.uns["dataset_level_QC"] = qc_summary

with open(
    AUDIT_DIR / "GSE182109_QC_summary.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(qc_summary, handle, indent=2)

print(qc_summary)

In [ ]:
temporary_output = OUTPUT_H5AD.with_suffix(".tmp.h5ad")
if temporary_output.exists():
    temporary_output.unlink()

adata_qc.write_h5ad(temporary_output, compression="gzip")
temporary_output.replace(OUTPUT_H5AD)

with sc.read_h5ad(OUTPUT_H5AD, backed="r") as check:
    if check.shape != EXPECTED_OUTPUT_SHAPE:
        raise AssertionError(
            f"Saved-file validation failed: observed {check.shape}"
        )
    if "counts" not in check.layers:
        raise AssertionError("Saved output is missing layers['counts'].")

print("Saved and validated:", OUTPUT_H5AD)